# C7-cnn-transfer — Practice p26

**Type:** challenge · **Difficulty:** advanced · **Concepts:** convolution, tensor-shape-tracing, cnn-training

**Time budget:** 75 minutes

## Part I — pure-Python floor-formula tracer (9 points)

Implement `trace_conv_stack(input_shape, specs)`, where `input_shape` is
`(B,C,H,W)` and each spec is
`(c_out,k_h,k_w,s_h,s_w,p_h,p_w)`. Return the output tuple after every
convolution. Use the dilation-1 floor formula independently on both axes and
raise `ValueError` if any computed spatial size is below 1.

Apply it to the exact supplied `(3,17,191,227)` input and three specs, assigning
the list to `trace`.

## Part II — helper-led construction and training (11 points)

Keep Part I pure Python. In the later training cell, use the completed helper
first to assign `small_trace` for the supplied input `(15,2,13,15)` and specs.
Only then construct the convolution/ReLU stack in the listed spec order, then construct adaptive pool `(1,1)`, flatten, and the 3-class linear head last.
Verify constructed shapes against the
already committed helper trace. Train the supplied seeded full batch for
exactly 20 SGD steps (`lr=0.12`) with `CrossEntropyLoss`, and assign a boolean
`training_certificate` requiring: exact trace agreement, logits `(15,3)`,
finite loss with final at most `0.75 * initial`, exact optimizer ownership,
all parameter gradients present, and at least one moved parameter.

**Banned — zero points for Part I:** importing `torch` or `torchvision`,
constructing/running a layer or model, `torch.nn.functional`, shape-calculator
APIs, model summaries, or hiding any such operation in a helper.

**Banned — zero points for Part II:** pretrained weights, downloads/network,
changing supplied data/seed/draw order/specs/batching/steps/hyperparameters,
computing `small_trace` from observed layer outputs, or reporting only loss
without trace/ownership/gradient/movement checks.

In [ ]:
def trace_conv_stack(input_shape, specs):
    # Part I: pure Python only.
    ...


input_shape = (3, 17, 191, 227)
specs = [
    (31, 7, 3, 2, 1, 3, 1),
    (43, 3, 5, 3, 2, 0, 2),
    (59, 1, 7, 1, 3, 0, 0),
]
trace = trace_conv_stack(input_shape, specs)


In [ ]:
# Part II begins here; torch was forbidden only in Part I.
import torch
import torch.nn as nn

SEED = 20260804
torch.set_default_dtype(torch.float64)
torch.manual_seed(SEED)
small_input_shape = (15, 2, 13, 15)
small_specs = [
    (4, 3, 5, 1, 2, 1, 2),
    (6, 3, 3, 2, 1, 1, 1),
]
small_trace = trace_conv_stack(small_input_shape, small_specs)
if not isinstance(small_trace, list) or len(small_trace) != len(small_specs):
    raise RuntimeError("commit a complete helper-produced small_trace first")

generator = torch.Generator(device="cpu").manual_seed(SEED)
train_X = 0.05 * torch.randn(*small_input_shape, generator=generator)
train_y = torch.arange(small_input_shape[0], dtype=torch.long) % 3
train_X[train_y == 0, :, :, 2:5] += 1.0
train_X[train_y == 1, :, 6:9, :] += 1.0
diagonal = torch.arange(13)
class_two = train_X[train_y == 2].clone()
class_two[:, :, diagonal, diagonal] += 1.0
train_X[train_y == 2] = class_two
train_X[train_y == 0] -= 0.8
train_X[train_y == 2] += 0.8


In [ ]:
# Construct only after small_trace exists, then verify and train.
# YOUR CODE HERE
features = ...
model = ...
constructed_trace_agrees = ...
loss_history = ...
optimizer_owns_exactly_model = ...
gradient_names = ...
moved_parameter_names = ...
training_certificate = ...
